In [1]:
import yfinance as yf
import pandas as pd
# Pull data for one company - let's use Apple as our test
ticker = yf.Ticker("AAPL")

In [2]:
# Get Apple's earnings call dates
earnings_dates = ticker.get_earnings_dates(limit=8)
earnings_dates

,EPS Estimate,Reported EPS,Surprise(%)
Earnings Date,,,
2026-07-30 16:00:00-04:00,1.89,NaN,NaN
2026-04-30 16:00:00-04:00,1.94,2.01,3.46
2026-01-29 16:00:00-05:00,2.67,2.84,6.34
2025-10-30 16:00:00-04:00,1.77,1.85,4.52
2025-07-31 16:00:00-04:00,1.43,1.57,10.12
2025-05-01 16:00:00-04:00,1.62,1.65,1.69
2025-01-30 16:00:00-05:00,2.34,2.40,2.52
2024-10-31 16:00:00-04:00,0.95,0.97,2.48
2024-08-01 16:00:00-04:00,1.34,1.40,4.36


In [3]:

# Pick one earnings date to test with
earnings_date = pd.Timestamp("2025-10-30")

# Pull a window of price data around that date
price_data = ticker.history(start="2025-10-27", end="2025-11-05")
price_data

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2025-10-27 00:00:00-04:00,264.132730,268.360759,263.903368,268.051636,44888200,0.0,0.0
2025-10-28 00:00:00-04:00,268.231107,269.128593,267.393481,268.241089,41534800,0.0,0.0
2025-10-29 00:00:00-04:00,268.520319,270.644315,266.356428,268.939148,51086700,0.0,0.0
2025-10-30 00:00:00-04:00,271.222670,273.366629,267.722593,270.634338,69886500,0.0,0.0
2025-10-31 00:00:00-04:00,276.208558,276.537644,268.400661,269.607239,86167100,0.0,0.0
2025-11-03 00:00:00-05:00,269.657146,270.085925,265.498896,268.290985,50194600,0.0,0.0
2025-11-04 00:00:00-05:00,267.573000,270.724089,266.865012,269.278198,49274800,0.0,0.0


In [4]:
#Closing price on the earnings call day
close_on_call = price_data.loc["2025-10-30","Close"]
#Closing price 1 trading day after
close_1day = price_data.loc["2025-10-31","Close"]
#Closing price 3 trading days after
close_3day = price_data.loc["2025-11-04","Close"]

#Calculating percentage price reaction
reaction_1day = (close_1day - close_on_call) / close_on_call * 100
reaction_3day = (close_3day - close_on_call) / close_on_call * 100
print(f"1-day reaction: {reaction_1day:.2f}%")
print(f"3-day reaction: {reaction_3day:.2f}%")

1-day reaction: -0.38%
3-day reaction: -0.50%


In [5]:
def get_price_reaction(ticker_symbol, earnings_date_str, days_before=5, days_after=10):
    """
    Given a ticker and an earnings date, returns the 1-day and 3-day
    percentage price reaction after that earnings call.
    """
    ticker = yf.Ticker(ticker_symbol)
    earnings_date = pd.Timestamp(earnings_date_str)

    start = earnings_date - pd.Timedelta(days=days_before)
    end = earnings_date + pd.Timedelta(days=days_after)
    price_data = ticker.history(start=start, end=end)

    # NEW: skip if no data came back at all
    if price_data.empty:
        return None

    price_data.index = price_data.index.tz_localize(None)

    trading_days = price_data.index[price_data.index >= earnings_date]

    if len(trading_days) < 4:
        return None

    call_day = trading_days[0]
    day_1 = trading_days[1]
    day_3 = trading_days[3]

    close_on_call = price_data.loc[call_day, "Close"]
    close_1day = price_data.loc[day_1, "Close"]
    close_3day = price_data.loc[day_3, "Close"]

    reaction_1day = (close_1day - close_on_call) / close_on_call * 100
    reaction_3day = (close_3day - close_on_call) / close_on_call * 100

    return {
        "ticker": ticker_symbol,
        "earnings_date": earnings_date_str,
        "reaction_1day": round(reaction_1day, 2),
        "reaction_3day": round(reaction_3day, 2)
    }

In [6]:
result = get_price_reaction("AAPL", "2025-10-30")
result

{'ticker': 'AAPL',
 'earnings_date': '2025-10-30',
 'reaction_1day': np.float64(-0.38),
 'reaction_3day': np.float64(-0.5)}

In [7]:
# let's try Microsoft
result_msft = get_price_reaction("MSFT", "2025-07-30")
result_msft

{'ticker': 'MSFT',
 'earnings_date': '2025-07-30',
 'reaction_1day': np.float64(3.95),
 'reaction_3day': np.float64(4.36)}

In [8]:
# Expanded list across sectors for variety
tickers = [
    # Tech
    "AAPL", "MSFT", "AMZN", "META", "NVDA",
    # Finance
    "JPM", "BAC", 
    # Retail/Consumer
    "WMT", "NKE",
    # Healthcare
    "UNH",
    # Industrial/Other
    "DIS"
]
print(f"Total companies: {len(tickers)}")
all_results = []

for symbol in tickers:
    ticker_obj = yf.Ticker(symbol)
    earnings = ticker_obj.get_earnings_dates(limit=12)
    earnings = earnings.dropna(subset=["Reported EPS"])  # keep only completed calls

    for date in earnings.index:
        date_str = date.strftime("%Y-%m-%d")
        result = get_price_reaction(symbol, date_str)
        if result is not None:
            # attach the earnings surprise info too
            result["surprise_pct"] = earnings.loc[date, "Surprise(%)"]
            all_results.append(result)

print(f"Collected {len(all_results)} earnings events total")

Total companies: 11
Collected 262 earnings events total


In [9]:
df = pd.DataFrame(all_results)
df.head(10)

,ticker,earnings_date,reaction_1day,reaction_3day,surprise_pct
0,AAPL,2026-04-30,3.24,4.73,3.46
1,AAPL,2026-01-29,0.46,4.34,6.34
2,AAPL,2025-10-30,-0.38,-0.50,4.52
3,AAPL,2025-07-31,-2.50,-2.24,10.12
4,AAPL,2025-05-01,-3.74,-6.94,1.69
5,AAPL,2025-01-30,-0.67,-2.02,2.52
6,AAPL,2024-10-31,-1.33,-1.09,2.48
7,AAPL,2024-08-01,0.69,-5.10,4.36
8,AAPL,2024-05-02,5.98,5.42,2.05
9,AAPL,2024-02-01,-0.54,1.31,3.55


In [10]:
df.to_csv("earnings_price_reactions.csv", index=False)

In [11]:
import os
import re
def parse_transcript_file(filepath):
    """
    Reads one transcript file and extracts ticker, year, quarter, and text.
    """
    filename = os.path.basename(filepath)
    # Extract year, quarter, ticker from filename like "2024_Q1_aapl_processed.txt"
    match = re.match(r"(\d{4})_Q(\d)_([a-zA-Z]+)_processed\.txt", filename)
    if not match:
        return None
    year, quarter, ticker = match.groups()
    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()
    return {
        "ticker": ticker.upper(),
        "year": int(year),
        "quarter": int(quarter),
        "filename": filename,
        "text": text
    }

In [12]:
test_file = "Cleaned_ECTs_Dataset/Apple/2024_Q1_aapl_processed.txt"
result = parse_transcript_file(test_file)
print(result["ticker"], result["year"], result["quarter"])
print(result["text"][:300])

AAPL 2024 1
Earnings Call AnalysisQ1-2024 AnalysisApple IncModest Revenue and iPhone Sales, Solid Services GrowthThis quarter, the company has maintained a steady course, with both total company revenue and iPhone revenue projected to be on par with last year's figures. Services business retains its momentum wi


In [13]:
company_folders = os.listdir("Cleaned_ECTs_Dataset")
company_folders

['.ipynb_checkpoints',
 'Amazon',
 'Apple',
 'BAC',
 'JPM',
 'Meta',
 'Microsoft',
 'Nike',
 'Nvidia',
 'UnitedHealth',
 'Walmart',
 'WaltDisney']

In [14]:
ticker_to_folder = {
    "AAPL": "Apple",
    "MSFT": "Microsoft",
    "AMZN": "Amazon",
    "META": "Meta",
    "NVDA": "Nvidia",
    "JPM": "JPM",
    "BAC": "BAC",
    "WMT": "Walmart",
    "NKE": "Nike",
    "UNH": "UnitedHealth",
    "DIS": "WaltDisney"
}

In [15]:
all_transcripts = []

for ticker_symbol, folder_name in ticker_to_folder.items():
    folder_path = f"Cleaned_ECTs_Dataset/{folder_name}"
    files = os.listdir(folder_path)
    
    for file in files:
        if file.endswith("_processed.txt"):
            full_path = f"{folder_path}/{file}"
            result = parse_transcript_file(full_path)
            if result is not None:
                all_transcripts.append(result)

print(f"Collected {len(all_transcripts)} transcripts total")

Collected 295 transcripts total


In [16]:
transcripts_df = pd.DataFrame(all_transcripts)
transcripts_df.head(5)

,ticker,year,quarter,filename,text
0,AAPL,2018,1,2018_Q1_aapl_processed.txt,0 Good day.At this time for opening remarks an...
1,AAPL,2018,2,2018_Q2_aapl_processed.txt,At this time for opening remarks and introduct...
2,AAPL,2018,3,2018_Q3_aapl_processed.txt,At this time for opening remarks and introduct...
3,AAPL,2018,4,2018_Q4_aapl_processed.txt,At this time for opening remarks and introduct...
4,AAPL,2019,1,2019_Q1_aapl_processed.txt,At this time for opening remarks and introduct...


In [17]:
transcripts_df.to_csv("earnings_transcripts.csv", index=False)

In [18]:
# Load our price reactions data fresh (in case kernel was restarted)
price_df = pd.read_csv("earnings_price_reactions.csv")

# Convert earnings_date to a proper date type
price_df["earnings_date"] = pd.to_datetime(price_df["earnings_date"])

# Extract year and quarter from the date
price_df["year"] = price_df["earnings_date"].dt.year
price_df["quarter"] = price_df["earnings_date"].dt.quarter

price_df.head(10)

,ticker,earnings_date,reaction_1day,reaction_3day,surprise_pct,year,quarter
0,AAPL,2026-04-30,3.24,4.73,3.46,2026,2
1,AAPL,2026-01-29,0.46,4.34,6.34,2026,1
2,AAPL,2025-10-30,-0.38,-0.50,4.52,2025,4
3,AAPL,2025-07-31,-2.50,-2.24,10.12,2025,3
4,AAPL,2025-05-01,-3.74,-6.94,1.69,2025,2
5,AAPL,2025-01-30,-0.67,-2.02,2.52,2025,1
6,AAPL,2024-10-31,-1.33,-1.09,2.48,2024,4
7,AAPL,2024-08-01,0.69,-5.10,4.36,2024,3
8,AAPL,2024-05-02,5.98,5.42,2.05,2024,2
9,AAPL,2024-02-01,-0.54,1.31,3.55,2024,1


In [19]:
transcripts_df = pd.read_csv("earnings_transcripts.csv")

merged_df = pd.merge(
    price_df,
    transcripts_df,
    on=["ticker", "year", "quarter"],
    how="inner"
)

print(f"Price records: {len(price_df)}")
print(f"Transcript records: {len(transcripts_df)}")
print(f"Matched records after merge: {len(merged_df)}")

Price records: 262
Transcript records: 295
Matched records after merge: 187


In [20]:
unmatched = pd.merge(
    price_df,
    transcripts_df,
    on=["ticker", "year", "quarter"],
    how="left",
    indicator=True
)

unmatched_only = unmatched[unmatched["_merge"] == "left_only"]
unmatched_only[["ticker", "earnings_date", "year", "quarter"]].sort_values("ticker")

,ticker,earnings_date,year,quarter
0,AAPL,2026-04-30,2026,2
1,AAPL,2026-01-29,2026,1
2,AAPL,2025-10-30,2025,4
3,AAPL,2025-07-31,2025,3
4,AAPL,2025-05-01,2025,2
...,...,...,...,...
169,WMT,2025-08-21,2025,3
168,WMT,2025-11-20,2025,4
167,WMT,2026-02-19,2026,1
166,WMT,2026-05-21,2026,2


In [21]:
real_mismatches = unmatched_only[unmatched_only["year"] < 2025]
print(f"Genuine mismatches (excluding 2025/2026 date gap): {len(real_mismatches)}")
real_mismatches[["ticker", "earnings_date", "year", "quarter"]].sort_values("ticker")

Genuine mismatches (excluding 2025/2026 date gap): 14


,ticker,earnings_date,year,quarter
6,AAPL,2024-10-31,2024,4
54,AMZN,2024-10-31,2024,4
55,AMZN,2024-08-01,2024,3
149,BAC,2024-10-15,2024,4
150,BAC,2024-07-16,2024,3
244,DIS,2024-11-14,2024,4
256,DIS,2021-11-10,2021,4
126,JPM,2024-10-11,2024,4
78,META,2024-10-30,2024,4
79,META,2024-07-31,2024,3


In [22]:
merged_df.to_csv("merged_dataset.csv", index=False)
print(f"Saved merged dataset with {len(merged_df)} records")
merged_df.head(5)

Saved merged dataset with 187 records


,ticker,earnings_date,reaction_1day,reaction_3day,surprise_pct,year,quarter,filename,text
0,AAPL,2024-08-01,0.69,-5.10,4.36,2024,3,2024_Q3_aapl_processed.txt,"8 billion, up 5% year-over-year.2 billion, gro..."
1,AAPL,2024-05-02,5.98,5.42,2.05,2024,2,2024_Q2_aapl_processed.txt,"8 billion, down 4% year-over-year, with an EPS..."
2,AAPL,2024-02-01,-0.54,1.31,3.55,2024,1,2024_Q1_aapl_processed.txt,Earnings Call AnalysisQ1-2024 AnalysisApple In...
3,AAPL,2023-11-02,-0.52,2.39,4.93,2023,4,2023_Q4_aapl_processed.txt,Earnings Call AnalysisQ4-2023 AnalysisApple In...
4,AAPL,2023-08-03,-4.80,-5.95,5.66,2023,3,2023_Q3_aapl_processed.txt,"At this time, for opening remarks and introduc..."


In [23]:
hedging_words = [
    "may", "might", "could", "possibly", "perhaps", "uncertain",
    "uncertainty", "approximately", "estimate", "estimates", "estimated",
    "believe", "believes", "expect", "expects", "anticipate", "anticipates",
    "likely", "unlikely", "risk", "risks", "cautious", "caution",
    "depend", "depends", "contingent", "assume", "assumes", "assumption"
]

print(f"Total hedging words in our list: {len(hedging_words)}")

Total hedging words in our list: 29


In [24]:
def count_hedging_words(text, word_list):
    """
    Counts how many times hedging words appear in a transcript,
    as a percentage of total words.
    """
    text_lower = text.lower()
    words = text_lower.split()
    total_words = len(words)
    
    hedge_count = 0
    for word in words:
        # strip punctuation like periods/commas stuck to words
        clean_word = word.strip(".,!?;:\"'()")
        if clean_word in word_list:
            hedge_count += 1
    
    hedging_pct = (hedge_count / total_words) * 100
    return round(hedging_pct, 3)

In [25]:
sample_text = merged_df.iloc[0]["text"]
hedging_score = count_hedging_words(sample_text, hedging_words)
print(f"Hedging language: {hedging_score}% of words")

Hedging language: 0.458% of words


In [26]:
for i in range(5):
    text = merged_df.iloc[i]["text"]
    ticker = merged_df.iloc[i]["ticker"]
    date = merged_df.iloc[i]["earnings_date"]
    score = count_hedging_words(text, hedging_words)
    print(f"{ticker} ({date}): {score}%")

AAPL (2024-08-01 00:00:00): 0.458%
AAPL (2024-05-02 00:00:00): 0.582%
AAPL (2024-02-01 00:00:00): 0.435%
AAPL (2023-11-02 00:00:00): 0.483%
AAPL (2023-08-03 00:00:00): 0.557%


In [27]:
hedging_scores = []

for text in merged_df["text"]:
    score = count_hedging_words(text, hedging_words)
    hedging_scores.append(score)

merged_df["hedging_pct"] = hedging_scores
merged_df[["ticker", "earnings_date", "hedging_pct"]].head(10)

,ticker,earnings_date,hedging_pct
0,AAPL,2024-08-01,0.458
1,AAPL,2024-05-02,0.582
2,AAPL,2024-02-01,0.435
3,AAPL,2023-11-02,0.483
4,AAPL,2023-08-03,0.557
5,AAPL,2023-05-04,0.542
6,AAPL,2023-02-02,0.414
7,AAPL,2022-10-27,0.565
8,AAPL,2022-07-28,0.773
9,AAPL,2022-04-28,0.685


In [28]:
print("Minimum hedging %:", merged_df["hedging_pct"].min())
print("Maximum hedging %:", merged_df["hedging_pct"].max())
print("Average hedging %:", merged_df["hedging_pct"].mean().round(3))

Minimum hedging %: 0.252
Maximum hedging %: 1.006
Average hedging %: 0.595


In [29]:
forward_looking_words = [
    "will", "expect", "expects", "expected", "expecting",
    "anticipate", "anticipates", "anticipated",
    "believe", "believes", "believed",
    "plan", "plans", "planned", "planning",
    "intend", "intends", "intended",
    "forecast", "forecasts", "forecasted",
    "project", "projects", "projected",
    "outlook", "guidance"
]

print(f"Total forward-looking words in our list: {len(forward_looking_words)}")

Total forward-looking words in our list: 26


In [30]:
sample_text = merged_df.iloc[0]["text"]
forward_score = count_hedging_words(sample_text, forward_looking_words)
print(f"Forward-looking language: {forward_score}% of words")

Forward-looking language: 0.773% of words


In [31]:
forward_scores = []

for text in merged_df["text"]:
    score = count_hedging_words(text, forward_looking_words)
    forward_scores.append(score)

merged_df["forward_looking_pct"] = forward_scores
merged_df[["ticker", "earnings_date", "hedging_pct", "forward_looking_pct"]].head(10)

,ticker,earnings_date,hedging_pct,forward_looking_pct
0,AAPL,2024-08-01,0.458,0.773
1,AAPL,2024-05-02,0.582,0.732
2,AAPL,2024-02-01,0.435,0.579
3,AAPL,2023-11-02,0.483,0.851
4,AAPL,2023-08-03,0.557,0.665
5,AAPL,2023-05-04,0.542,0.596
6,AAPL,2023-02-02,0.414,0.691
7,AAPL,2022-10-27,0.565,0.523
8,AAPL,2022-07-28,0.773,0.787
9,AAPL,2022-04-28,0.685,0.618


In [32]:
print("Minimum forward-looking %:", merged_df["forward_looking_pct"].min())
print("Maximum forward-looking %:", merged_df["forward_looking_pct"].max())
print("Average forward-looking %:", merged_df["forward_looking_pct"].mean().round(3))

Minimum forward-looking %: 0.408
Maximum forward-looking %: 1.476
Average forward-looking %: 0.838


In [33]:
import textstat

sample_text = merged_df.iloc[0]["text"]
readability_score = textstat.flesch_reading_ease(sample_text)
print(f"Flesch Reading Ease score: {readability_score}")

Flesch Reading Ease score: 55.26121926008926


In [34]:
readability_scores = []

for text in merged_df["text"]:
    score = textstat.flesch_reading_ease(text)
    readability_scores.append(score)

merged_df["readability_score"] = readability_scores
merged_df[["ticker", "earnings_date", "hedging_pct", "forward_looking_pct", "readability_score"]].head(10)

,ticker,earnings_date,hedging_pct,forward_looking_pct,readability_score
0,AAPL,2024-08-01,0.458,0.773,55.261219
1,AAPL,2024-05-02,0.582,0.732,56.995563
2,AAPL,2024-02-01,0.435,0.579,55.415637
3,AAPL,2023-11-02,0.483,0.851,57.456830
4,AAPL,2023-08-03,0.557,0.665,58.362505
5,AAPL,2023-05-04,0.542,0.596,57.044901
6,AAPL,2023-02-02,0.414,0.691,57.970087
7,AAPL,2022-10-27,0.565,0.523,58.519046
8,AAPL,2022-07-28,0.773,0.787,57.352474
9,AAPL,2022-04-28,0.685,0.618,58.845807


In [35]:
print("Minimum readability score:", merged_df["readability_score"].min())
print("Maximum readability score:", merged_df["readability_score"].max())
print("Average readability score:", merged_df["readability_score"].mean().round(2))

Minimum readability score: 43.86455169813584
Maximum readability score: 72.56132554168006
Average readability score: 54.98


In [36]:
merged_df.to_csv("merged_dataset_with_features.csv", index=False)

In [37]:
from transformers import pipeline
sentiment_model = pipeline("sentiment-analysis", model="ProsusAI/finbert")

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

C:\Users\Aradhya\miniconda3\envs\earnings-sentiment\lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Aradhya\.cache\huggingface\hub\models--ProsusAI--finbert. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [43]:
sample_text = merged_df.iloc[0]["text"][:500]  # just first 500 characters for now
result = sentiment_model(sample_text)
result

[{'label': 'positive', 'score': 0.9575027823448181}]

In [44]:
def chunk_text(text, chunk_size=500):
    """
    Splits a long text into smaller chunks of roughly chunk_size words each.
    """
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
    return chunks

In [46]:
def get_average_sentiment(text, model, chunk_size=500):
    """
    Splits text into chunks, scores each with FinBERT, and returns
    an average sentiment score from -1 (negative) to +1 (positive).
    """
    chunks = chunk_text(text, chunk_size)
    
    scores = []
    for chunk in chunks:
        result = model(chunk, truncation=True)[0]
        label = result["label"]
        confidence = result["score"]
        
        # Convert to a signed score: positive stays positive, negative becomes negative
        if label == "positive":
            scores.append(confidence)
        elif label == "negative":
            scores.append(-confidence)
        else:  # neutral
            scores.append(0)
    
    average_score = sum(scores) / len(scores)
    return round(average_score, 4)

In [47]:
sample_text = merged_df.iloc[0]["text"]
score = get_average_sentiment(sample_text, sentiment_model)
print(f"Average sentiment score: {score}")

Average sentiment score: 0.3152


In [48]:
import time
start_time = time.time()

test_scores = []
for i in range(10):
    text = merged_df.iloc[i]["text"]
    score = get_average_sentiment(text, sentiment_model)
    test_scores.append(score)

end_time = time.time()
print(f"Time for 10 transcripts: {round(end_time - start_time, 1)} seconds")
print(test_scores)

Time for 10 transcripts: 71.8 seconds
[0.3152, 0.1234, 0.194, 0.0862, 0.125, -0.07, -0.004, 0.1805, -0.048, 0.2055]


In [49]:
sentiment_scores = []
for i in range(len(merged_df)):
    text = merged_df.iloc[i]["text"]
    score = get_average_sentiment(text, sentiment_model)
    sentiment_scores.append(score)

merged_df["sentiment_score"] = sentiment_scores

In [50]:
merged_df.to_csv("merged_dataset_final.csv", index=False)

In [51]:
merged_df[["ticker", "earnings_date", "hedging_pct", "forward_looking_pct", "readability_score", "sentiment_score"]].head(10)

,ticker,earnings_date,hedging_pct,forward_looking_pct,readability_score,sentiment_score
0,AAPL,2024-08-01,0.458,0.773,55.261219,0.3152
1,AAPL,2024-05-02,0.582,0.732,56.995563,0.1234
2,AAPL,2024-02-01,0.435,0.579,55.415637,0.1940
3,AAPL,2023-11-02,0.483,0.851,57.456830,0.0862
4,AAPL,2023-08-03,0.557,0.665,58.362505,0.1250
5,AAPL,2023-05-04,0.542,0.596,57.044901,-0.0700
6,AAPL,2023-02-02,0.414,0.691,57.970087,-0.0040
7,AAPL,2022-10-27,0.565,0.523,58.519046,0.1805
8,AAPL,2022-07-28,0.773,0.787,57.352474,-0.0480
9,AAPL,2022-04-28,0.685,0.618,58.845807,0.2055


In [52]:
print("Sentiment - Min:", merged_df["sentiment_score"].min())
print("Sentiment - Max:", merged_df["sentiment_score"].max())
print("Sentiment - Average:", merged_df["sentiment_score"].mean().round(3))

Sentiment - Min: -0.2026
Sentiment - Max: 0.5032
Sentiment - Average: 0.17


In [53]:
def get_market_reaction(earnings_date_str, days_before=5, days_after=10):
    """
    Same logic as get_price_reaction, but for the S&P 500 index,
    to use as a market return control variable.
    """
    return get_price_reaction("^GSPC", earnings_date_str, days_before, days_after)

In [54]:
test_market = get_market_reaction("2024-08-01")
test_market

{'ticker': '^GSPC',
 'earnings_date': '2024-08-01',
 'reaction_1day': np.float64(-1.84),
 'reaction_3day': np.float64(-3.79)}

In [55]:
unique_dates = merged_df["earnings_date"].unique()
print(f"Number of unique earnings dates: {len(unique_dates)}")

Number of unique earnings dates: 159


In [56]:
market_results = []
for date in unique_dates:
    result = get_market_reaction(date)
    if result is not None:
        market_results.append(result)
print(f"Collected {len(market_results)} market reactions")

Collected 159 market reactions


In [58]:
market_df = pd.DataFrame(market_results)
market_df = market_df.rename(columns={
    "reaction_1day": "market_reaction_1day",
    "reaction_3day": "market_reaction_3day"
})
market_df = market_df.drop(columns=["ticker"])  # drop the "^GSPC" ticker column, not needed
market_df.head(5)

,earnings_date,market_reaction_1day,market_reaction_3day
0,2024-08-01,-1.84,-3.79
1,2024-05-02,1.26,2.44
2,2024-02-01,1.07,0.98
3,2023-11-02,0.94,1.40
4,2023-08-03,-0.53,-0.06


In [60]:
final_df = pd.merge(merged_df, market_df, on="earnings_date", how="left")
final_df[["ticker", "earnings_date", "reaction_1day", "market_reaction_1day", "surprise_pct"]].head(10)

,ticker,earnings_date,reaction_1day,market_reaction_1day,surprise_pct
0,AAPL,2024-08-01,0.69,-1.84,4.36
1,AAPL,2024-05-02,5.98,1.26,2.05
2,AAPL,2024-02-01,-0.54,1.07,3.55
3,AAPL,2023-11-02,-0.52,0.94,4.93
4,AAPL,2023-08-03,-4.80,-0.53,5.66
5,AAPL,2023-05-04,4.69,1.85,6.37
6,AAPL,2023-02-02,2.44,-1.04,-3.78
7,AAPL,2022-10-27,7.56,2.46,1.44
8,AAPL,2022-07-28,3.28,1.42,3.93
9,AAPL,2022-04-28,-3.66,-3.63,6.48


In [61]:
final_df.to_csv("final_dataset.csv", index=False)
final_df.columns.tolist()

['ticker',
 'earnings_date',
 'reaction_1day',
 'reaction_3day',
 'surprise_pct',
 'year',
 'quarter',
 'filename',
 'text',
 'hedging_pct',
 'forward_looking_pct',
 'readability_score',
 'sentiment_score',
 'market_reaction_1day',
 'market_reaction_3day']

In [62]:
correlation_columns = ["reaction_1day", "reaction_3day", "surprise_pct", 
                        "hedging_pct", "forward_looking_pct", "readability_score", 
                        "sentiment_score", "market_reaction_1day", "market_reaction_3day"]

correlation_matrix = final_df[correlation_columns].corr()
correlation_matrix

,reaction_1day,reaction_3day,surprise_pct,hedging_pct,forward_looking_pct,readability_score,sentiment_score,market_reaction_1day,market_reaction_3day
reaction_1day,1.000000,0.925032,0.153144,0.125588,0.018079,-0.029158,0.032020,0.458446,0.195555
reaction_3day,0.925032,1.000000,0.140278,0.127824,0.067715,-0.039016,0.006198,0.379148,0.326615
surprise_pct,0.153144,0.140278,1.000000,0.015927,0.042098,-0.006334,0.002522,0.083292,0.082047
hedging_pct,0.125588,0.127824,0.015927,1.000000,0.496653,0.017261,-0.294597,0.085570,0.136700
forward_looking_pct,0.018079,0.067715,0.042098,0.496653,1.000000,-0.400135,-0.043609,-0.058742,0.137821
readability_score,-0.029158,-0.039016,-0.006334,0.017261,-0.400135,1.000000,-0.324842,0.107437,-0.046153
sentiment_score,0.032020,0.006198,0.002522,-0.294597,-0.043609,-0.324842,1.000000,-0.102823,-0.084161
market_reaction_1day,0.458446,0.379148,0.083292,0.085570,-0.058742,0.107437,-0.102823,1.000000,0.586735
market_reaction_3day,0.195555,0.326615,0.082047,0.136700,0.137821,-0.046153,-0.084161,0.586735,1.000000


In [63]:
import statsmodels.api as sm
X = final_df[["sentiment_score", "hedging_pct", "forward_looking_pct", 
              "readability_score", "surprise_pct", "market_reaction_1day"]]
y = final_df["reaction_1day"]
X = sm.add_constant(X)

model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:          reaction_1day   R-squared:                       0.246
Model:                            OLS   Adj. R-squared:                  0.221
Method:                 Least Squares   F-statistic:                     9.795
Date:                Thu, 16 Jul 2026   Prob (F-statistic):           2.45e-09
Time:                        00:00:39   Log-Likelihood:                -596.35
No. Observations:                 187   AIC:                             1207.
Df Residuals:                     180   BIC:                             1229.
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                    1.4076 